In [ ]:
"""
Data paths assume the standard Kaggle dataset mount:
  /kaggle/input/datasets/rrickyroger/movielens-1m/
"""

# ── Standard imports ──────────────────────────────────────────────────────────
import re, copy, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


# ══════════════════════════════════════════════════════════════════════════════
#  1. CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════

GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir",
    "Horror", "Musical", "Mystery", "Romance", "Sci-Fi",
    "Thriller", "War", "Western",
]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
NUM_GENRES   = 18

EMBED_DIM    = 32
USER_DIM     = EMBED_DIM + 1 + EMBED_DIM + EMBED_DIM   # 97
MOVIE_DIM    = EMBED_DIM                                # 32
FUSION_DIM   = USER_DIM + MOVIE_DIM                     # 129

ZIP_VOCAB    = 3500
OCC_VOCAB    = 21
AGE_MAX      = 56.0

SUPPORT_SIZE = 20

INNER_LR     = 5e-6   # α  — paper §4.1 for MovieLens
OUTER_LR     = 5e-5   # β  — paper §4.1 for MovieLens
INNER_STEPS  = 5      # paper tests 1–5; 5 is the reported best
META_BATCH   = 32     # paper §4.1
EPOCHS       = 30     # paper §4.1

LOG_INTERVAL = 50

DATA_ROOT   = "/kaggle/input/datasets/rrickyroger/movielens-1m"
OUTPUT_ROOT = "/kaggle/working"


# ══════════════════════════════════════════════════════════════════════════════
#  2. MODEL
# ══════════════════════════════════════════════════════════════════════════════

class UserEmbedder(nn.Module):
    """
    Maps user attributes → user vector (θ₁ in the paper).
    Gender + age + occupation + zip_code, matching Figure 2.
    """
    def __init__(self, embed_dim=EMBED_DIM, age_max=AGE_MAX):
        super().__init__()
        self.age_max    = age_max
        self.gender_emb = nn.Embedding(2,         embed_dim)
        self.occ_emb    = nn.Embedding(OCC_VOCAB,  embed_dim)
        self.zip_emb    = nn.Embedding(ZIP_VOCAB,  embed_dim)
        for emb in (self.gender_emb, self.occ_emb, self.zip_emb):
            nn.init.normal_(emb.weight, 0.0, 0.01)

    def forward(self, gender, age, occupation, zipcode):
        g = self.gender_emb(gender)
        a = (age / self.age_max).unsqueeze(1)
        o = self.occ_emb(occupation)
        z = self.zip_emb(zipcode)
        return torch.cat([g, a, o, z], dim=1)   # (B, 97)


class MovieEmbedder(nn.Module):
    """
    Maps movie genre multi-hot → item vector (part of θ₁ in the paper).
    NOTE: The paper also uses year, director, and actor from IMDb. This
    implementation keeps only genre for simplicity on the raw MovieLens-1M
    dataset (no IMDb augmentation). If you add those features, replace this
    Linear projection with a concatenation of several embeddings, matching
    Figure 2 more closely.
    """
    def __init__(self, num_genres=NUM_GENRES, embed_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(num_genres, embed_dim, bias=True)
        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, genre_multihot):
        return self.proj(genre_multihot)         # (B, 32)


class RatingMLP(nn.Module):
    """
    Decision-making / output network (θ₂ in the paper, Eq. 3).
    This is the ONLY part adapted in the inner loop.
    """
    def __init__(self, in_dim=FUSION_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256,    128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128,     64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64,      32), nn.ReLU(),
            # ── CHANGE: Output layer is now linear (no sigmoid) ──────────────
            # Original code applied sigmoid(raw) * 4 + 1 to force output into
            # [1, 5]. The paper (§3.1, Eq. 3) says "a linear function might be
            # appropriate" for rating estimation. Sigmoid saturates near 1 and 5,
            # producing near-zero gradients for extreme-rated movies and making
            # it harder for the model to learn from 1-star and 5-star samples.
            # We use a raw linear output here and clamp to [1, 5] only at
            # inference / evaluation time (see forward() below).
            nn.Linear(32, 1),
        )
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.net(x).squeeze(1)   # (B,)  unconstrained during training


class MovieRecommender(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_embedder  = UserEmbedder()
        self.movie_embedder = MovieEmbedder()
        self.mlp            = RatingMLP()

    def forward(self, gender, age, occupation, zipcode, genre_multihot):
        user_vec  = self.user_embedder(gender, age, occupation, zipcode)
        movie_vec = self.movie_embedder(genre_multihot)
        fused     = torch.cat([user_vec, movie_vec], dim=1)
        return self.mlp(fused)


# ══════════════════════════════════════════════════════════════════════════════
#  3. DATASET
# ══════════════════════════════════════════════════════════════════════════════

def _hash_zip(z):
    digits = re.sub(r"\D", "", str(z))
    return int(digits[:5]) % ZIP_VOCAB if digits else 0

def _gender_int(g):
    return 1 if str(g).strip().upper() == "F" else 0

def _multihot(genre_str):
    vec = torch.zeros(NUM_GENRES)
    for g in genre_str.split("|"):
        idx = GENRE_TO_IDX.get(g.strip())
        if idx is not None:
            vec[idx] = 1.0
    return vec

def _make_episode(rows, user_feat, movie_feat):
    g, a, o, z = user_feat
    genres, ratings = [], []
    for mid, rat in rows:
        mf = movie_feat.get(mid)
        if mf is None:
            continue
        genres.append(mf)
        ratings.append(rat)
    n = len(ratings)
    if n == 0:
        return None
    return {
        "gender":         torch.full((n,), g, dtype=torch.long),
        "age":            torch.full((n,), a, dtype=torch.float),
        "occupation":     torch.full((n,), o, dtype=torch.long),
        "zipcode":        torch.full((n,), z, dtype=torch.long),
        "genre_multihot": torch.stack(genres),
        "rating":         torch.tensor(ratings, dtype=torch.float),
    }


class MovieLensMAML(Dataset):
    """
    Builds per-user tasks (support, query) following the MeLU paper.
      1. Query set is fixed at build time (QUERY_SIZE_AT_BUILD items per user)
      2. Users are split 80 / 20 into train_tasks / test_tasks so that the
         20% held-out users are true "new users" never seen during meta-training
         (cold-start evaluation, per paper §4).
    """
    def __init__(self, users_path, movies_path, ratings_path,
                 support_size=SUPPORT_SIZE,
                 # ── CHANGE: query_size now a first-class parameter ──────────
                 # Original code built a variable-length query pool and
                 # sub-sampled it at training time. Paper fixes query size
                 # = 10 at dataset construction (§4.1). We make it explicit.
                 query_size=QUERY_SIZE_AT_BUILD,
                 seed=42):
        super().__init__()
        self.support_size = support_size
        self.query_size   = query_size
        rng = random.Random(seed)

        users_df   = pd.read_csv(users_path,   sep="::", engine="python", header=None,
                                  names=["user_id","gender","age","occupation","zip_code"],
                                  encoding="latin-1")
        movies_df  = pd.read_csv(movies_path,  sep="::", engine="python", header=None,
                                  names=["movie_id","title","genres"],
                                  encoding="latin-1")
        ratings_df = pd.read_csv(ratings_path, sep="::", engine="python", header=None,
                                  names=["user_id","movie_id","rating","timestamp"],
                                  encoding="latin-1")

        user_feat = {}
        for _, r in users_df.iterrows():
            user_feat[int(r["user_id"])] = (
                _gender_int(r["gender"]),
                float(r["age"]),
                int(r["occupation"]),
                _hash_zip(r["zip_code"]),
            )

        movie_feat = {}
        for _, r in movies_df.iterrows():
            movie_feat[int(r["movie_id"])] = _multihot(r["genres"])

        self.user_feat  = user_feat
        self.movie_feat = movie_feat
        all_tasks       = []

        for uid, grp in ratings_df.groupby("user_id"):
            if uid not in user_feat:
                continue
            rows = list(zip(grp["movie_id"].tolist(), grp["rating"].tolist()))
            rng.shuffle(rows)

            if len(rows) < support_size + query_size:
                continue
            support_rows = rows[:support_size]
            query_rows   = rows[support_size : support_size + query_size]

            # Sanity check: support and query must never share a movie
            sup_mids = {mid for mid, _ in support_rows}
            qry_mids = {mid for mid, _ in query_rows}
            assert sup_mids.isdisjoint(qry_mids), (
                f"User {uid}: support/query overlap detected!"
            )

            all_tasks.append((int(uid), support_rows, query_rows))


        rng.shuffle(all_tasks)
        cut = int(0.8 * len(all_tasks))
        self.train_tasks = all_tasks[:cut]
        self.test_tasks  = all_tasks[cut:]
        self.tasks       = self.train_tasks   # __getitem__ indexes train by default

        print(f"[Dataset] {len(self.train_tasks)} train users | "
              f"{len(self.test_tasks)} test (cold-start) users | "
              f"support={support_size} | query={query_size} (fixed)")

    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        uid, sup_rows, qry_rows = self.tasks[idx]
        uf  = self.user_feat[uid]
        sup = _make_episode(sup_rows, uf, self.movie_feat)
        qry = _make_episode(qry_rows, uf, self.movie_feat)
        return sup, qry


def collate_tasks(batch):
    supports = [b[0] for b in batch if b[0] is not None and b[1] is not None]
    queries  = [b[1] for b in batch if b[0] is not None and b[1] is not None]
    return supports, queries


# ══════════════════════════════════════════════════════════════════════════════
#  4. MAML CORE
# ══════════════════════════════════════════════════════════════════════════════

def batch_to_device(d, device):
    return {k: v.to(device) for k, v in d.items()}

def forward_with_params(model, mlp_params, batch):
    """Forward pass using functional (per-step) MLP params + global embedders."""
    from torch.func import functional_call
    user_vec  = model.user_embedder(
        batch["gender"], batch["age"], batch["occupation"], batch["zipcode"])
    movie_vec = model.movie_embedder(batch["genre_multihot"])
    fused     = torch.cat([user_vec, movie_vec], dim=1)
    pred      = functional_call(model.mlp, mlp_params, (fused,))
    return pred.squeeze(-1)

def inner_loop(model, support, inner_lr, inner_steps):
    """
    Adapt θ₂ (MLP only) on the support set via gradient descent.
    Returns θ′ — the adapted MLP parameters for this user.
    Embedder weights (θ₁) are NOT touched here; they only receive
    gradient updates through the outer (meta) loss.
    """
    local_params = {n: p.clone() for n, p in model.mlp.named_parameters()}
    for _ in range(inner_steps):
        pred  = forward_with_params(model, local_params, support)
        loss  = F.l1_loss(pred, support["rating"])
        grads = torch.autograd.grad(
            loss, list(local_params.values()),
            create_graph=True, allow_unused=True)
        local_params = {
            name: param - inner_lr * (grad if grad is not None else torch.zeros_like(param))
            for (name, param), grad in zip(local_params.items(), grads)
        }
    return local_params


# ══════════════════════════════════════════════════════════════════════════════
#  5. TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

def meta_train():
    dataset = MovieLensMAML(
        users_path   = f"{DATA_ROOT}/users.dat",
        movies_path  = f"{DATA_ROOT}/movies.dat",
        ratings_path = f"{DATA_ROOT}/ratings.dat",
    )

    train_dataset = dataset   # dataset.__getitem__ already indexes train_tasks
    loader = DataLoader(train_dataset, batch_size=META_BATCH, shuffle=True,
                        collate_fn=collate_tasks, num_workers=2)

    model    = MovieRecommender().to(DEVICE)
    meta_opt = torch.optim.Adam(model.parameters(), lr=OUTER_LR)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}\n")
    print(f"{'Epoch':>5} {'Step':>6} {'Meta-MAE':>10} {'MAE':>8}  Time/step")
    print("─" * 55)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, batch_count = 0.0, 0
        t0 = time.time()

        for supports, queries in loader:
            if not supports:
                continue
            meta_opt.zero_grad()

            total_loss, valid = torch.tensor(0.0, device=DEVICE), 0

            for sup_raw, qry_raw in zip(supports, queries):
                sup = batch_to_device(sup_raw, DEVICE)
                qry = batch_to_device(qry_raw, DEVICE)

                if sup["rating"].numel() == 0 or qry["rating"].numel() == 0:
                    continue

                # Inner loop: adapt θ₂ on support set
                theta_prime = inner_loop(model, sup, INNER_LR, INNER_STEPS)

                loss_q     = F.l1_loss(pred_q, qry["rating"])
                total_loss = total_loss + loss_q
                valid     += 1

            if valid == 0:
                continue

            avg_loss = total_loss / valid
            avg_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            meta_opt.step()

            running_loss += avg_loss.item()
            batch_count  += 1

            if batch_count % LOG_INTERVAL == 0:
                ml  = running_loss / batch_count
                ms  = (time.time() - t0) / batch_count * 1000
                print(f"{epoch:>5} {batch_count:>6} {ml:>10.4f} {ml:>8.4f}  {ms:>7.1f}ms")

        if batch_count:
            ml = running_loss / batch_count
            print(f"\n  ▶ Epoch {epoch} │ meta-MAE={ml:.4f}\n")
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "opt_state": meta_opt.state_dict(), "meta_loss": ml,
            }, f"{OUTPUT_ROOT}/ckpt_epoch{epoch}.pt")

    print("✅ Training done.")
    return model, dataset


# ══════════════════════════════════════════════════════════════════════════════
#  6. MODEL SAVE / LOAD
# ══════════════════════════════════════════════════════════════════════════════

def save_model(model, path, meta=None):
    """Save GLOBAL model weights (embedders + MLP) for reuse later."""
    payload = {
        "model_state_dict": model.state_dict(),
        "model_class":      "MovieRecommender",
        "meta":             meta or {},
    }
    torch.save(payload, path)
    print(f"✅ Model saved → {path}")


def load_model(path, device=DEVICE):
    """Load a model saved with save_model()."""
    payload = torch.load(path, map_location=device)
    model = MovieRecommender().to(device)
    model.load_state_dict(payload["model_state_dict"])
    print(f"✅ Model loaded from {path}  (meta: {payload.get('meta', {})})")
    return model


# ══════════════════════════════════════════════════════════════════════════════
#  7. EVIDENCE CANDIDATE SELECTION  (new — paper §3.3)
# ══════════════════════════════════════════════════════════════════════════════

def compute_evidence_candidates(global_model, dataset, movies_df,
                                 top_k=10, device=DEVICE):
    """
    Implements MeLU §3.3: rank all movies by
        popularity × avg Frobenius-norm of ∇_θ₂ loss over training users.

    Returns a DataFrame of the top_k candidate movies to show new users.
    """
    global_model.eval()

    # Count how many training users rated each movie (popularity proxy)
    movie_count = {}
    for uid, sup_rows, qry_rows in dataset.train_tasks:
        for mid, _ in sup_rows + qry_rows:
            movie_count[mid] = movie_count.get(mid, 0) + 1

    # Accumulate gradient norms per movie across all training users
    movie_grad_norm = {}
    movie_user_count = {}

    n_users = len(dataset.train_tasks)
    print(f"[EvidenceCandidates] Computing gradient norms over {n_users} users...")

    for step, (uid, sup_rows, _) in enumerate(dataset.train_tasks):
        uf = dataset.user_feat[uid]
        ep = _make_episode(sup_rows, uf, dataset.movie_feat)
        if ep is None:
            continue
        ep = batch_to_device(ep, device)

        # We need per-item gradient norms: compute loss for each item separately
        for item_idx in range(ep["rating"].shape[0]):
            single = {k: v[item_idx].unsqueeze(0) for k, v in ep.items()}
            mid = sup_rows[item_idx][0]

            # Zero out any accumulated grads
            global_model.zero_grad()

            pred = global_model(
                single["gender"], single["age"],
                single["occupation"], single["zipcode"],
                single["genre_multihot"])
            loss = F.l1_loss(pred, single["rating"])
            loss.backward()

            # Frobenius norm of gradients across θ₂ (MLP) parameters
            frob = sum(
                p.grad.norm(p="fro").item() ** 2
                for p in global_model.mlp.parameters()
                if p.grad is not None
            ) ** 0.5

            movie_grad_norm[mid]   = movie_grad_norm.get(mid, 0.0) + frob
            movie_user_count[mid]  = movie_user_count.get(mid, 0) + 1

        if (step + 1) % 500 == 0:
            print(f"  processed {step+1}/{n_users} users...")

    # Average gradient norm over users that rated each movie
    scores = {}
    for mid in movie_grad_norm:
        avg_norm   = movie_grad_norm[mid] / movie_user_count[mid]
        popularity = movie_count.get(mid, 0)
        scores[mid] = popularity * avg_norm

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    # Build a readable DataFrame
    title_map  = dict(zip(movies_df["movie_id"], movies_df["title"]))
    genres_map = dict(zip(movies_df["movie_id"], movies_df["genres"]))
    rows = []
    for rank, (mid, score) in enumerate(ranked, 1):
        rows.append({
            "rank":       rank,
            "movie_id":   mid,
            "title":      title_map.get(mid, f"movie_{mid}"),
            "genres":     genres_map.get(mid, ""),
            "popularity": movie_count.get(mid, 0),
            "avg_grad_norm": movie_grad_norm[mid] / movie_user_count[mid],
            "evidence_score": score,
        })
    df = pd.DataFrame(rows)
    print(f"\n[EvidenceCandidates] Top-{top_k} evidence candidates:")
    print(df[["rank", "title", "genres", "popularity", "evidence_score"]].to_string(index=False))
    return df


# ══════════════════════════════════════════════════════════════════════════════
#  8. MELU-STYLE PER-USER EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

OCC_LABELS = {
     0: "other / not specified",  1: "academic / educator",   2: "artist",
     3: "clerical / admin",       4: "college / grad student", 5: "customer service",
     6: "doctor / health care",   7: "executive / managerial", 8: "farmer",
     9: "homemaker",             10: "K-12 student",          11: "lawyer",
    12: "programmer",            13: "retired",               14: "sales / marketing",
    15: "scientist",             16: "self-employed",         17: "technician / engineer",
    18: "tradesman / craftsman", 19: "unemployed",            20: "writer",
}
AGE_LABELS = {1: "Under 18", 18: "18-24", 25: "25-34", 35: "35-44",
              45: "45-49", 50: "50-55", 56: "56+"}


def personalize_for_user(global_model, support_batch, finetune_steps=5, lr=1e-2):
    """
    Simulates MeLU's meta-test procedure for ONE new user:
      - deep-copies the GLOBAL model (global_model is never mutated)
      - fine-tunes ONLY the MLP (θ₂) on the support set
      - returns the personalized model + loss trajectory
    """
    local_model = copy.deepcopy(global_model).to(DEVICE)
    local_model.train()

    optimizer = torch.optim.SGD(local_model.mlp.parameters(), lr=lr)

    loss_trace = []
    for _ in range(finetune_steps):
        optimizer.zero_grad()
        pred = local_model(
            support_batch["gender"], support_batch["age"],
            support_batch["occupation"], support_batch["zipcode"],
            support_batch["genre_multihot"],
        )
        # ── CHANGE: Fine-tune loss is MAE, matching the paper's objective ─────
        # Original: loss = F.mse_loss(pred, support_batch["rating"])
        loss = F.l1_loss(pred, support_batch["rating"])
        loss.backward()
        optimizer.step()
        loss_trace.append(loss.item())

    local_model.eval()
    return local_model, loss_trace


def param_delta_report(global_model, local_model):
    """Per-layer L2 distance between personalized MLP and global MLP."""
    report = []
    g_params = dict(global_model.mlp.named_parameters())
    l_params = dict(local_model.mlp.named_parameters())
    for name in g_params:
        delta = (l_params[name].detach() - g_params[name].detach()).norm().item()
        gnorm = g_params[name].detach().norm().item()
        report.append((name, delta, gnorm))
    return report


def _bar(rating, width=20):
    filled = int(round(max(0.0, min(rating, 5.0)) / 5.0 * width))
    return "█" * filled + "░" * (width - filled)


def print_user_card(uid, user_feat, users_raw):
    g, a, o, z = user_feat
    gender_str = "Female" if g == 1 else "Male"
    age_str    = AGE_LABELS.get(int(a), str(int(a)))
    occ_str    = OCC_LABELS.get(o, f"code {o}")
    row = users_raw[users_raw["user_id"] == uid]
    zip_str = row["zip_code"].values[0] if len(row) else "N/A"
    print(f"\n{'═'*72}\n  USER {uid}\n{'═'*72}")
    print(f"  Gender     : {gender_str}")
    print(f"  Age group  : {age_str}")
    print(f"  Occupation : {occ_str}")
    print(f"  Zip code   : {zip_str}")


def print_support_set(sup_rows, movie_title, movie_genres):
    print(f"\n  ┌─ SUPPORT SET ({len(sup_rows)} movies used to fine-tune this user's model) ──")
    print(f"  │  {'#':>2}  {'Rating':>6}  {'Title':<42}  Genres")
    print(f"  │  {'─'*2}  {'─'*6}  {'─'*42}  {'─'*20}")
    for i, (mid, rat) in enumerate(sup_rows, 1):
        title  = movie_title.get(mid, f"movie_{mid}")[:41]
        genres = movie_genres.get(mid, "")
        print(f"  │  {i:>2}  {rat:>5.1f}★  {title:<42}  {genres}")
    print(f"  └{'─'*70}")


def print_predictions(pred_rows, movie_title, movie_genres, top_k=20):
    top = pred_rows[:top_k]
    print(f"\n  ┌─ TOP {top_k} RECOMMENDATIONS (personalized model, query pool) ──────────")
    print(f"  │  {'#':>2}  {'Pred':>5}  {'Bar':<20}  {'Actual':>6}  {'Title':<38}  Genres")
    print(f"  │  {'─'*2}  {'─'*5}  {'─'*20}  {'─'*6}  {'─'*38}  {'─'*18}")
    for rank, (mid, pred, actual) in enumerate(top, 1):
        title  = movie_title.get(mid, f"movie_{mid}")[:37]
        genres = movie_genres.get(mid, "")[:30]
        bar    = _bar(pred)
        print(f"  │  {rank:>2}  {pred:>4.2f}  {bar}  {actual:>5.1f}★  {title:<38}  {genres}")
    print(f"  └{'─'*70}")


def print_param_report(report, finetune_steps, loss_trace):
    print(f"\n  ┌─ PERSONALIZED PARAMETERS (MLP only, {finetune_steps} fine-tune steps) ──")
    print(f"  │  Loss trajectory: {['%.4f' % l for l in loss_trace]}")
    print(f"  │")
    print(f"  │  {'Layer':<22} {'‖Δθ‖ (moved from global)':>26}  {'‖θ_global‖':>12}")
    print(f"  │  {'─'*22} {'─'*26}  {'─'*12}")
    for name, delta, gnorm in report:
        pct = (delta / gnorm * 100) if gnorm > 0 else 0.0
        print(f"  │  {name:<22} {delta:>20.4f} ({pct:>4.1f}%)  {gnorm:>12.4f}")
    print(f"  └{'─'*70}")


def run_melu_evaluation(global_model, dataset, movie_title, movie_genres, users_raw,
                         n_users=5, top_k=20, finetune_steps=5, finetune_lr=1e-2, seed=99):
    """
    MeLU meta-test evaluation on HELD-OUT new users (test_tasks only).

    For n_users randomly chosen from dataset.test_tasks:
      global θ → clone → fine-tune on support (disjoint from query)
               → evaluate / recommend on query pool → report

    global_model is NEVER modified — only deep copies are fine-tuned.
    """
    global_model.eval()

    eval_pool = dataset.test_tasks
    rng       = random.Random(seed)
    indices   = rng.sample(range(len(eval_pool)), min(n_users, len(eval_pool)))
    all_maes, user_reports = [], []

    for pick_num, idx in enumerate(indices, 1):
        uid, sup_rows, qry_rows = eval_pool[idx]
        user_feat = dataset.user_feat[uid]

        sup_mids = {mid for mid, _ in sup_rows}
        qry_mids = {mid for mid, _ in qry_rows}
        assert sup_mids.isdisjoint(qry_mids), f"User {uid}: support/query overlap!"

        def make_ep(rows):
            g, a, o, z = user_feat
            mids, genre_vecs, ratings = [], [], []
            for mid, rat in rows:
                mf = dataset.movie_feat.get(mid)
                if mf is None:
                    continue
                mids.append(mid)
                genre_vecs.append(mf)
                ratings.append(rat)
            n = len(ratings)
            if n == 0:
                return None, []
            batch = {
                "gender":         torch.full((n,), g, dtype=torch.long),
                "age":            torch.full((n,), a, dtype=torch.float),
                "occupation":     torch.full((n,), o, dtype=torch.long),
                "zipcode":        torch.full((n,), z, dtype=torch.long),
                "genre_multihot": torch.stack(genre_vecs),
                "rating":         torch.tensor(ratings, dtype=torch.float),
            }
            return {k: v.to(DEVICE) for k, v in batch.items()}, mids

        sup_batch, _            = make_ep(sup_rows)
        qry_batch, qry_mids_list = make_ep(qry_rows)
        if sup_batch is None or qry_batch is None:
            continue

        # Clone global θ, fine-tune on support (MeLU meta-test)
        local_model, loss_trace = personalize_for_user(
            global_model, sup_batch, finetune_steps=finetune_steps, lr=finetune_lr)

        # Evaluate on disjoint query set
        with torch.no_grad():
            preds = local_model(
                qry_batch["gender"], qry_batch["age"], qry_batch["occupation"],
                qry_batch["zipcode"], qry_batch["genre_multihot"])

        mae  = F.l1_loss(preds, actuals).item()
        rmse = F.mse_loss(preds, actuals).item() ** 0.5
        all_maes.append(mae)

        order     = preds.argsort(descending=True).tolist()
        pred_rows = [(qry_mids_list[i], preds[i].item(), actuals[i].item()) for i in order]
        param_report = param_delta_report(global_model, local_model)

        print(f"\n\n{'#'*72}\n  MELU META-TEST  │  USER {pick_num} of {n_users}  (user_id={uid})  [COLD-START]\n{'#'*72}")
        print_user_card(uid, user_feat, users_raw)
        print_support_set(sup_rows, movie_title, movie_genres)
        print_predictions(pred_rows, movie_title, movie_genres, top_k=top_k)
        print_param_report(param_report, finetune_steps, loss_trace)
        # ── CHANGE: Report both MAE (primary) and RMSE (secondary) ───────────
        print(f"\n  Query-set MAE  (personalized model, paper metric): {mae:.4f}")
        print(f"  Query-set RMSE (secondary):                         {rmse:.4f}")
        print(f"  Support/query disjoint: ✅ "
              f"({len(sup_mids)} support, {len(qry_mids)} query, 0 overlap)")
        print(f"  User is from HELD-OUT test split (true cold-start): ✅")

        user_reports.append({
            "user_id": uid, "mae": mae, "rmse": rmse,
            "loss_trace": loss_trace,
            "param_report": param_report,
            "personalized_model": local_model,
        })

    avg_mae = sum(all_maes) / len(all_maes) if all_maes else float("nan")
    print(f"\n{'═'*72}")
    print(f"  OVERALL │ {len(all_maes)} cold-start users │ avg MAE = {avg_mae:.4f}")
    print(f"{'═'*72}\n")
    return user_reports, avg_mae


# ══════════════════════════════════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    model, dataset = meta_train()

    # Save final global model
    save_model(model, f"{OUTPUT_ROOT}/recommender_maml_final.pt",
               meta={"epochs": EPOCHS, "inner_steps": INNER_STEPS,
                     "inner_lr": INNER_LR, "outer_lr": OUTER_LR})

    # Build title / genre / user lookups for pretty printing
    movies_df    = pd.read_csv(f"{DATA_ROOT}/movies.dat", sep="::", engine="python",
                                header=None, names=["movie_id","title","genres"],
                                encoding="latin-1")
    movie_title  = dict(zip(movies_df["movie_id"], movies_df["title"]))
    movie_genres = dict(zip(movies_df["movie_id"], movies_df["genres"]))
    users_raw    = pd.read_csv(f"{DATA_ROOT}/users.dat", sep="::", engine="python",
                                header=None,
                                names=["user_id","gender","age","occupation","zip_code"],
                                encoding="latin-1")

    # ── Section 7: Evidence candidate selection (paper §3.3) ─────────────────
    # This was entirely missing from the original code.
    # It identifies the best movies to show new users during cold-start onboarding.
    print("\n── MeLU §3.3: Evidence candidate selection ──")
    evidence_df = compute_evidence_candidates(
        global_model = model,
        dataset      = dataset,
        movies_df    = movies_df,
        top_k        = 10,
    )

    # ── Section 8: Per-user evaluation on cold-start (held-out) users ────────
    print("\n── MeLU-style per-user personalisation & evaluation (cold-start users) ──")
    run_melu_evaluation(
        global_model  = model,
        dataset        = dataset,
        movie_title    = movie_title,
        movie_genres   = movie_genres,
        users_raw      = users_raw,
        n_users        = 5,
        top_k          = 20,
        finetune_steps = 5,
        finetune_lr    = 1e-2,
    )